# Question 1: datetime Fundamentals and Time Series Indexing

This question focuses on datetime handling and time series indexing using patient vital signs data.

## Setup

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('default')
sns.set_style('whitegrid')

# Create output directory
os.makedirs('output', exist_ok=True)

## Part 1.1: Load and Explore Data

**Note:** This dataset contains realistic healthcare data characteristics:
- **200 patients** with daily vital signs over 1 year
- **Missing visits**: Patients miss approximately 5% of scheduled visits (realistic!)
- **Different start dates**: Not all patients start monitoring on January 1st (some join later)
- When selecting data by date ranges, you may find that some patients don't have data for certain periods - this is expected and realistic

In [12]:
# Load patient vital signs data
patient_vitals = pd.read_csv('data/patient_vitals.csv')

print("Patient vitals shape:", patient_vitals.shape)
print("Patient vitals columns:", patient_vitals.columns.tolist())

# Display sample data
print("\nPatient vitals sample:")
print(patient_vitals.head())
print("\nData summary:")
print(patient_vitals.describe())

# Check date range and missing data patterns
print(f"\nDate range: {patient_vitals['date'].min()} to {patient_vitals['date'].max()}")
print(f"Unique patients: {patient_vitals['patient_id'].nunique()}")
print(f"Total records: {len(patient_vitals)}")
print(f"Expected records (200 patients × 365 days): {200 * 365:,}")
print(f"Missing visits: ~{200 * 365 - len(patient_vitals):,} records")

Patient vitals shape: (18250, 7)
Patient vitals columns: ['date', 'patient_id', 'temperature', 'heart_rate', 'blood_pressure_systolic', 'blood_pressure_diastolic', 'weight']

Patient vitals sample:
         date patient_id  temperature  heart_rate  blood_pressure_systolic  \
0  2023-01-01      P0001    98.389672          71                      119   
1  2023-01-02      P0001    98.492046          67                      117   
2  2023-01-03      P0001    98.790163          70                      113   
3  2023-01-04      P0001    98.635781          74                      117   
4  2023-01-05      P0001    98.051660          67                      118   

   blood_pressure_diastolic     weight  
0                        84  68.996865  
1                        82  67.720215  
2                        78  67.846825  
3                        82  67.693993  
4                        83  68.228852  

Data summary:
        temperature    heart_rate  blood_pressure_systolic  \
count  182

## Part 1.2: datetime Operations

**TODO: Perform datetime operations**

In [13]:
# TODO: Convert date column to datetime
# patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
print("1. Converted date column to datetime")
patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
print(f"Loaded {len(patient_vitals):,} records\n")

# TODO: Set datetime column as index
# patient_vitals = patient_vitals.set_index('date')
print("2. Set datetime column as index")
patient_vitals = patient_vitals.set_index('date')
print(f" Index type: {type(patient_vitals).index}")
print(f" Index name: {patient_vitals.index.name}\n")

# TODO: Extract year, month, day components from datetime index
print("3. Extracted year, month, day components from datetime index\n")
# patient_vitals['year'] = None  # Extract from index
patient_vitals['year'] = patient_vitals.index.year
# patient_vitals['month'] = None  # Extract from index
patient_vitals['month'] = patient_vitals.index.month
# patient_vitals['day'] = None  # Extract from index
patient_vitals['day'] = patient_vitals.index.day



# TODO: Calculate time differences (e.g., days since first measurement)
# Note: Since patients start at different times, calculate days_since_start per patient
# Hint: To use groupby on the 'date' column, temporarily reset the index, then set it back
# Example: patient_vitals_reset = patient_vitals.reset_index()
#          Use groupby('patient_id')['date'].transform(lambda x: (x - x.min()).dt.days)
#          Or use groupby('patient_id').apply() to calculate days from each patient's first date
#          Then: patient_vitals = patient_vitals_reset.set_index('date')
# patient_vitals['days_since_start'] = None  # Calculate from each patient's start date

# First, reset the index temporarily
patient_vitals_reset = patient_vitals.reset_index()
# Calculate days since each patient's first measurement with groupby...
patient_vitals_reset['days_since_start'] = patient_vitals_reset.groupby('patient_id')['date'].transform(
    lambda x: (x - x.min()).dt.days
)
# Set index back to date
patient_vitals = patient_vitals_reset.set_index('date')
print("4. Calculate time difference since start for each patient")
print(f" Days since start calculated for {patient_vitals['patient_id'].nunique()} patients")
print(f" Range: {patient_vitals['days_since_start'].min()} to {patient_vitals['days_since_start'].max()} days\n")


# TODO: Create business day ranges for clinic visit schedules
# clinic_dates = None  # Use pd.bdate_range() for clinic visits
start_date = patient_vitals.index.min()
end_date = patient_vitals.index.max()

clinic_dates = pd.bdate_range(start = start_date, end = end_date, freq = 'B')
print("5. Business day ranges for clinic visit schedules")
print("f Clinic business days: {len(clinic_dates)} days")
print("f First lcinic day: {clinic_dates[0]}")
print("f Last clinic day: {clinic_dates [=1]}\n")

# TODO: Create date ranges with different frequencies
print("6. Create date ranges with different frequencies")
# daily_range = None  # Daily monitoring schedule
daily_range = pd.date_range(start=start_date, end=end_date, freq = 'D')
print(f" Daily schedule: {len(daily_range)} days")
# weekly_range = None  # Weekly lab test schedule (Mondays)
weekly_range = pd.date_range(start=start_date, end=end_date, freq = 'W-MON')
print(f" Weekly schedule (Mondays): {len(weekly_range)} weeks")
# monthly_range = None  # Monthly checkup schedule
monthly_range = pd.date_range(start=start_date, end=end_date, freq = 'MS')
print(f" Monthly schedule: {len(monthly_range)} months\n")

# TODO: Use date ranges to analyze visit patterns
print("7. Analyzing visit patterns")
# Check how many patient visits occurred on clinic business days vs weekends
# patient_dates_set = set(patient_vitals.index.date)
patient_dates_set = set(patient_vitals.index.date)
# clinic_dates_set = set(clinic_dates.date)
clinic_dates_set = set(clinic_dates.date)
# visits_on_clinic_days = len(patient_dates_set & clinic_dates_set)
visits_on_clinic_days = len(patient_dates_set & clinic_dates_set)
# visits_on_weekends = len(patient_dates_set) - visits_on_clinic_days
visits_on_weekends = len(patient_dates_set) - visits_on_clinic_days
# print(f"Visits on clinic business days: {visits_on_clinic_days}")
print(f"Visits on clinic business days: {visits_on_clinic_days}")
# print(f"Visits on weekends: {visits_on_weekends}")
print(f"Visits on weekends: {visits_on_weekends}")
# print(f"Total unique visit dates: {len(patient_dates_set)}")
print(f"Total unique visit dates: {len(patient_dates_set)}\n")

# TODO: Save results as 'output/q1_datetime_analysis.csv'
print("9. Saving datetime analysis results as 'output/q1_datetime_analysis.csv")
# Create a DataFrame with datetime analysis results including:
# - date (datetime index or column)
# - year, month, day (extracted from datetime)
# - days_since_start (calculated time differences)
# - patient_id
# - At least one original column (e.g., temperature, heart_rate)
datetime_analysis = patient_vitals[[
    'patient_id',
    'year',
    'month',
    'day',
    'days_since_start',
    'temperature',
    'heart_rate',
    'blood_pressure_systolic',
    'weight',
]].copy()
# Note: When saving to CSV with index=False, you'll need to convert the index to a column first
datetime_analysis = datetime_analysis.reset_index()

datetime_analysis.to_csv('output/q1_datetime_analysis.csv', index = False)
# Example structure:
# datetime_analysis = patient_vitals[['patient_id', 'year', 'month', 'day', 'days_since_start', 'temperature']].copy()
# datetime_analysis.to_csv('output/q1_datetime_analysis.csv', index=False)
print(f" Saved: {'output/q1_datetime_analysis.csv'}")
print(f" Columns: {list(datetime_analysis.columns)}")
print(f" Rows: {len(datetime_analysis):,}")


1. Converted date column to datetime
Loaded 18,250 records

2. Set datetime column as index
 Index type: <pandas._libs.properties.AxisProperty object at 0x113c71a80>
 Index name: date

3. Extracted year, month, day components from datetime index

4. Calculate time difference since start for each patient
 Days since start calculated for 50 patients
 Range: 0 to 364 days

5. Business day ranges for clinic visit schedules
f Clinic business days: {len(clinic_dates)} days
f First lcinic day: {clinic_dates[0]}
f Last clinic day: {clinic_dates [=1]}

6. Create date ranges with different frequencies
 Daily schedule: 365 days
 Weekly schedule (Mondays): 52 weeks
 Monthly schedule: 12 months

7. Analyzing visit patterns
Visits on clinic business days: 260
Visits on weekends: 105
Total unique visit dates: 365

9. Saving datetime analysis results as 'output/q1_datetime_analysis.csv
 Saved: output/q1_datetime_analysis.csv
 Columns: ['date', 'patient_id', 'year', 'month', 'day', 'days_since_start', 

## Part 1.3: Time Zone Handling

**TODO: Handle time zones**

In [23]:
# TODO: Create timezone-aware datetime (for multi-site clinical trials)
print("1. Create timezone-aware datetime (for multi-site clinical trials)")
# utc_time = None  # Current time in UTC
utc_time = pd.Timestamp.now(tz = 'UTC')
# eastern_time = None  # Convert to US Eastern
eastern_time = pd.Timestamp.now(tz = 'US/Eastern')
print("Current time across different trial site:")
print(f" UTC (Coordinated Universal Time): {utc_time}")
print(f" US/Eastern: {eastern_time}\n")

# Check if 'date' is already the index and reset if needed
if patient_vitals.index.name == 'date':
    print("-- Date is already the index. Resetting to column --")
    patient_vitals = patient_vitals.reset_index()

# patient_vitals['date'] = pd.to_datetime(patient_vitals['date'])
# print(f"Loaded {len(patient_vitals):,} records")
# print(f"Original date range: {patient_vitals['date'].min()} to {patient_vitals['date'].max()}")
# print(f"Original timezone: {patient_vitals['date'].dt.tz} (None = naive/no timezone)\n")

# TODO: Convert between different timezones
print("2. Converting patient data into timezone-aware format")
# Create timezone-aware DataFrame from patient_vitals
# patient_vitals_tz = None  # Localize to UTC
print("Localizing datetime to UTC")
patient_vitals_tz = patient_vitals.copy()
patient_vitals_tz['date'] = patient_vitals_tz['date'].dt.tz_localize('UTC')
print(f" Timezone after localization: {patient_vitals_tz['date'].dt.tz}")
# patient_vitals_tz_eastern = None  # Convert to Eastern time
print("Localizing datetime to US/Eastern time")
patient_vitals_tz_eastern = patient_vitals_tz.copy()
patient_vitals_tz_eastern['date'] = patient_vitals_tz_eastern['date'].dt.tz_convert('US/Eastern')
print(f" Timezone after conversion: {patient_vitals_tz_eastern['date'].dt.tz}")

# TODO: Handle daylight saving time transitions
# Create datetime that spans DST transition
print("3. Daylight saving time (DST) transitioning\n")
# Note: Using UTC avoids DST ambiguity issues - UTC has no daylight saving time
# Best practice: Store data in UTC, convert to local timezones only when needed
# dst_date_utc = pd.Timestamp('2023-03-12 10:00:00', tz='UTC')  # UTC time avoids DST issues
dst_date_utc = pd.Timestamp('2023-03-12 10:00:00', tz='UTC')
# dst_time_eastern = dst_date_utc.tz_convert('US/Eastern')  # Convert UTC to Eastern
dst_time_eastern = dst_date_utc.tz_convert('US/Eastern')

# TODO: Document timezone operations
# Create a report string with the following sections:
# 1. Original timezone: Describe what timezone your original data was in (or if it was naive)
# 2. Localization method: Explain how you localized the data (e.g., tz_localize('UTC'))
# 3. Conversion: Describe what timezone you converted to (e.g., 'US/Eastern')
# 4. DST handling: Document any issues or observations about daylight saving time transitions
#    Note: Explain why using UTC as the base timezone avoids DST ambiguity issues
# 5. Example: Show at least one example of a datetime before and after conversion
# Minimum length: 200 words

timezone_report = """
TODO: Document your timezone operations:
- What timezone was your original data in?
- How did you localize the data?
- What timezone did you convert to?
- What issues did you encounter with DST? (Note: Using UTC avoids DST ambiguity)
- Include at least one example showing a datetime before and after conversion
- Explain why UTC is recommended as the base timezone for storing temporal data
"""

print("="*80)
print("4. Comprehensive Timezone Operations Report")
print("="*80)
print()

report = """
Comprehensive Timezone Operations Report

1. ORIGINAL TIMEZONE
The original patient vitals data contained datetime values with no associated timezone information. 

2. LOCALIZATION
I localized the datetime data to UTC (Coordinated Universal Time) using tz_localize(), since UTC is the universal reference point for timezone conversions and does not change with daylight saving time. Localizing to UTC is the best practice of storing date data. 
The syntax to complete this was: 
patient_vitals_tz['date'] = patient_vitals_tz['date'].dt.tz_localize('UTC')
This assigns a timezone to each timestamp datapoint and maintains the original input date and time information

3. CONVERSION
I converted the data to US/Eastern time (from UTC) for sites in East Coast locations. Converting the dates to a different timezone preserves the absolute time, but adjusts the displayed time. This is beneficial in that we can interpret data in a local context and make sure that dates and times are consistent across sites in multiple locations.

4. DST
Thankfully, converting to UTC avoids DST ambiguity. DST creates confusion particularly during DST transition periods: in Spring, clocks move forward one hour, effectively eliminating an hour of recorded time. In Fall, clocks move back one our, creating an hour that occurs twice. These two annual events can create potential data loss or potential data duplication. UTC never changes for DST and thus, it is a stable reference. 

5. Example:

Original (naive):      2023-01-15 10:00:00
After localization:    2023-01-15 10:00:00+00:00 (UTC)
Converted to Eastern:  2023-01-15 05:00:00-05:00 (EST, UTC-5)

An example date with DST in summer:
UTC timestamp:         2023-07-15 14:00:00+00:00
US/Eastern:           2023-07-15 10:00:00-04:00 (EDT, UTC-4)
"""

print(report)

# TODO: Save results as 'output/q1_timezone_report.txt'
# with open('output/q1_timezone_report.txt', 'w') as f:
#     f.write(timezone_report)
with open('output/q1_timezone_report.txt', 'w') as f:
    f.write(report)

print("\n Saved: output/q1_timezone_report.txt")
print()

1. Create timezone-aware datetime (for multi-site clinical trials)
Current time across different trial site:
 UTC (Coordinated Universal Time): 2025-11-16 18:16:39.954573+00:00
 US/Eastern: 2025-11-16 13:16:39.954806-05:00

2. Converting patient data into timezone-aware format
Localizing datetime to UTC
 Timezone after localization: UTC
Localizing datetime to US/Eastern time
 Timezone after conversion: US/Eastern
3. Daylight saving time (DST) transitioning

4. Comprehensive Timezone Operations Report


Comprehensive Timezone Operations Report

1. ORIGINAL TIMEZONE
The original patient vitals data contained datetime values with no associated timezone information. 

2. LOCALIZATION
I localized the datetime data to UTC (Coordinated Universal Time) using tz_localize(), since UTC is the universal reference point for timezone conversions and does not change with daylight saving time. Localizing to UTC is the best practice of storing date data. 
The syntax to complete this was: 
patient_vital

## Submission Checklist

Before moving to Question 2, verify you've created:

- [ ] `output/q1_datetime_analysis.csv` - datetime analysis results
- [ ] `output/q1_timezone_report.txt` - timezone handling report
